In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
from statsmodels.stats.multitest import multipletests
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

import sys
!{sys.executable} -m pip install GEOparse -q
import GEOparse

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 130

print('All libraries imported successfully!')
print(f'GEOparse version: {GEOparse.__version__}')

All libraries imported successfully!
GEOparse version: 2.0.4


In [ ]:
print('Fetching GSE26927 from NCBI GEO...')
print('(Downloads ~50 MB on first run — subsequent runs use cache)')

gse = GEOparse.get_GEO(geo='GSE26927', destdir='./', silent=True)

print(f'\nDataset loaded : {gse.name}')
print(f'Title          : {gse.metadata["title"][0]}')
print(f'Total samples  : {len(gse.gsms)}')
print(f'\nFirst 5 sample IDs:')
for gsm_id in list(gse.gsms.keys())[:5]:
    print(f'  {gsm_id}: {gse.gsms[gsm_id].metadata["title"][0]}')

Fetching GSE26927 from NCBI GEO...
(Downloads ~50 MB on first run — subsequent runs use cache)


In [ ]:
# Build expression matrix
expr_dict = {}
for gsm_id, gsm in gse.gsms.items():
    if gsm.table is not None and not gsm.table.empty:
        expr_dict[gsm_id] = gsm.table.set_index('ID_REF')['VALUE']

expr_df = pd.DataFrame(expr_dict)
expr_df = expr_df.apply(pd.to_numeric, errors='coerce').dropna()
print(f'Expression matrix: {expr_df.shape[0]:,} probes × {expr_df.shape[1]} samples')

# Extract sample labels
MS_KEYWORDS      = ['ms', 'multiple sclerosis', 'patient', 'rrms', 'spms', 'ppms']
HEALTHY_KEYWORDS = ['healthy', 'control', 'normal', 'hc']

labels = {}
for gsm_id, gsm in gse.gsms.items():
    if gsm_id not in expr_df.columns:
        continue
    combined = ' '.join([
        gsm.metadata.get('title', [''])[0],
        gsm.metadata.get('source_name_ch1', [''])[0],
        *gsm.metadata.get('characteristics_ch1', [])
    ]).lower()

    if any(w in combined for w in HEALTHY_KEYWORDS):
        labels[gsm_id] = 'Healthy'
    elif any(w in combined for w in MS_KEYWORDS):
        labels[gsm_id] = 'MS'
    else:
        labels[gsm_id] = 'Unknown'

label_series = pd.Series(labels)
print(f'\nLabel counts:')
print(label_series.value_counts().to_string())

# Keep only clearly labelled samples
keep = [s for s in expr_df.columns if labels.get(s) in ['MS', 'Healthy']]
expr_df      = expr_df[keep]
label_series = label_series[keep]
print(f'\nRetained {len(keep)} labelled samples for analysis')

In [ ]:
# Log2 transform if needed 
median_val = expr_df.values.flatten()
median_val = np.nanmedian(median_val)
if median_val > 100:
    expr_df = np.log2(expr_df + 1)
    print(f'Applied log2(x+1) transformation  (median was {median_val:.1f})')
else:
    print(f'Data already log-transformed       (median = {median_val:.2f})')

# Variance filter — keep top 25% most variable probes
gene_var        = expr_df.var(axis=1)
var_threshold   = gene_var.quantile(0.75)          # top 25%
expr_filtered   = expr_df[gene_var >= var_threshold]

print(f'\nProbes before filtering : {expr_df.shape[0]:>6,}')
print(f'Probes after filtering  : {expr_filtered.shape[0]:>6,}  (top 25% by variance)')
print(f'Expression range        : {expr_filtered.values.min():.2f} – {expr_filtered.values.max():.2f}')
print(f'Mean expression         : {expr_filtered.values.mean():.2f}')

In [ ]:
# PCA
X        = expr_filtered.T
X_scaled = StandardScaler().fit_transform(X)

pca      = PCA(n_components=10, random_state=42)
pca_all  = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(pca_all[:, :2], columns=['PC1', 'PC2'], index=X.index)
pca_df['Group'] = label_series[pca_df.index].values

colors  = {'MS': '#E05C5C', 'Healthy': '#5C9BE0'}
markers = {'MS': 'o',       'Healthy': 's'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: scatter
ax = axes[0]
for group in ['Healthy', 'MS']:
    sub = pca_df[pca_df['Group'] == group]
    ax.scatter(sub['PC1'], sub['PC2'],
               c=colors[group], marker=markers[group],
               alpha=0.8, s=60, label=group,
               edgecolors='white', linewidth=0.5)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=11)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=11)
ax.set_title('PCA — MS vs Healthy Controls', fontsize=12)
ax.legend(fontsize=10)

# Right: scree plot
ax2 = axes[1]
pcs = range(1, 11)
ax2.bar(pcs, pca.explained_variance_ratio_ * 100, color='#7CB9E8', edgecolor='white')
ax2.plot(pcs, np.cumsum(pca.explained_variance_ratio_ * 100),
         color='#E05C5C', marker='o', linewidth=1.5, markersize=4, label='Cumulative')
ax2.set_xlabel('Principal Component', fontsize=11)
ax2.set_ylabel('Variance Explained (%)', fontsize=11)
ax2.set_title('Scree Plot', fontsize=12)
ax2.legend(fontsize=10)
ax2.set_xticks(list(pcs))

plt.tight_layout()
plt.savefig('pca_ms.png', bbox_inches='tight')
plt.show()
print('Saved: pca_ms.png')
print(f'PC1+PC2 variance explained: {sum(pca.explained_variance_ratio_[:2])*100:.1f}%')

In [ ]:
ms_samples      = label_series[label_series == 'MS'].index
healthy_samples = label_series[label_series == 'Healthy'].index

ms_expr      = expr_filtered[ms_samples]
healthy_expr = expr_filtered[healthy_samples]

print(f'MS samples      : {len(ms_samples)}')
print(f'Healthy samples : {len(healthy_samples)}')
print('Running Welch t-tests...')

# Vectorised t-test
t_stats, pvals = stats.ttest_ind(
    ms_expr.values, healthy_expr.values,
    axis=1, equal_var=False
)
log2fc = ms_expr.mean(axis=1).values - healthy_expr.mean(axis=1).values

# FDR correction (Benjamini-Hochberg)
_, padj, _, _ = multipletests(pvals, method='fdr_bh')

results = pd.DataFrame({
    'probe'       : expr_filtered.index,
    'log2FC'      : log2fc,
    'pval'        : pvals,
    'padj'        : padj,
    '-log10padj'  : -np.log10(np.clip(padj, 1e-300, None))
})

# Classify significance
results['significance'] = 'Not significant'
results.loc[(results['padj'] < 0.05) & (results['log2FC'] >  0.5), 'significance'] = 'Upregulated in MS'
results.loc[(results['padj'] < 0.05) & (results['log2FC'] < -0.5), 'significance'] = 'Downregulated in MS'

sig_counts = results['significance'].value_counts()
print(f'\nResults:')
print(f"  Upregulated in MS   : {sig_counts.get('Upregulated in MS',   0):>5,}")
print(f"  Downregulated in MS : {sig_counts.get('Downregulated in MS', 0):>5,}")
print(f"  Not significant     : {sig_counts.get('Not significant',     0):>5,}")

In [ ]:
color_map = {
    'Upregulated in MS'  : '#E05C5C',
    'Downregulated in MS': '#5C9BE0',
    'Not significant'    : '#CCCCCC'
}

fig, ax = plt.subplots(figsize=(10, 8))

for sig, color in color_map.items():
    sub   = results[results['significance'] == sig]
    alpha = 0.4 if sig == 'Not significant' else 0.8
    size  = 6   if sig == 'Not significant' else 20
    ax.scatter(sub['log2FC'], sub['-log10padj'],
               c=color, alpha=alpha, s=size,
               label=f'{sig} (n={len(sub):,})',
               edgecolors='none', rasterized=True)

# Threshold lines
ax.axhline(-np.log10(0.05), color='black', linestyle='--', linewidth=0.8, alpha=0.5, label='FDR = 0.05')
ax.axvline( 0.5, color='grey', linestyle=':', linewidth=0.8, alpha=0.7)
ax.axvline(-0.5, color='grey', linestyle=':', linewidth=0.8, alpha=0.7)

# Label top 10 most significant probes (non-overlapping)
top10 = results[results['significance'] != 'Not significant'].nlargest(10, '-log10padj')
texts = []
for _, row in top10.iterrows():
    txt = ax.text(row['log2FC'], row['-log10padj'], row['probe'],
                  fontsize=7.5, fontweight='bold', color='#333333')
    texts.append(txt)

try:
    from adjustText import adjust_text
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='grey', lw=0.5))
except ImportError:
    for txt in texts:
        txt.set_position((txt.get_position()[0] + 0.05, txt.get_position()[1] + 0.3))

ax.set_xlabel('log₂ Fold Change (MS vs Healthy)', fontsize=12)
ax.set_ylabel('−log₁₀ adjusted p-value', fontsize=12)
ax.set_title('Volcano Plot — Differential Gene Expression in MS', fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='upper left', framealpha=0.8)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('volcano_ms.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: volcano_ms.png')

In [ ]:
# ── Select top 40 DEGs ───────────────────────────────────────────────────────
sig_degs = results[results['significance'] != 'Not significant']

if len(sig_degs) >= 5:
    top_degs = sig_degs.nlargest(40, '-log10padj')['probe'].tolist()
else:
    # Fallback: top absolute fold change
    top_degs = results.assign(abs_fc=results['log2FC'].abs()) \
                       .nlargest(40, 'abs_fc')['probe'].tolist()
    print('Note: using top |fold-change| genes (few passed FDR threshold)')

# ── Prepare heatmap matrix ────────────────────────────────────────────────────
sample_order = [s for s in list(healthy_samples) + list(ms_samples)
                if s in expr_filtered.columns]
heatmap_data = expr_filtered.loc[top_degs, sample_order]

# Z-score each probe for visual contrast
from scipy.stats import zscore
heatmap_z = heatmap_data.apply(zscore, axis=1)

# Column colour bar
col_colors = pd.Series(
    ['#5C9BE0' if label_series[s] == 'Healthy' else '#E05C5C' for s in sample_order],
    index=sample_order
)

# ── Draw clustermap (no figsize kwarg — use height/col_ratio instead) ─────────
g = sns.clustermap(
    heatmap_z,
    col_colors=col_colors,
    col_cluster=False,
    row_cluster=True,
    cmap='RdBu_r',
    center=0,
    vmin=-2, vmax=2,
    xticklabels=False,
    yticklabels=True,
    figsize=(14, 10),
    cbar_kws={'label': 'Z-score (log₂ expression)'}
)
g.ax_heatmap.set_ylabel('Probe', fontsize=10)
g.fig.suptitle('Top Differentially Expressed Genes — MS vs Healthy',
               y=1.01, fontsize=13, fontweight='bold')

# Legend
healthy_patch = mpatches.Patch(color='#5C9BE0', label='Healthy')
ms_patch      = mpatches.Patch(color='#E05C5C', label='MS')
g.ax_col_colors.legend(
    handles=[healthy_patch, ms_patch],
    loc='upper right', bbox_to_anchor=(1.12, 2.5),
    frameon=False, fontsize=10
)

plt.savefig('heatmap_ms.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: heatmap_ms.png')